# Build the APOGEE DR17 × Gaia DR3 XP training set

This notebook constructs the labeled metadata table, applies explicit APOGEE quality cuts, resolves Gaia DR3 source IDs, checks XP-continuous availability, and optionally downloads XP coefficients in resumable batches.

**Important:** every sample count is printed from the data actually loaded. No counts are hard-coded.

# Section 1: imports and configuration

In [1]:
from pathlib import Path
import time
import json
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.table import Table, vstack
from astroquery.gaia import Gaia

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "src":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
BATCH_DIR = DATA_DIR / "apogee_xp_batches"
DATA_DIR.mkdir(exist_ok=True)
BATCH_DIR.mkdir(exist_ok=True)

ALLSTAR_PATH = DATA_DIR / "allStarLite-dr17-synspec_rev1.fits"
ALLSTAR_URL = "https://data.sdss.org/sas/dr17/apogee/spectro/aspcap/dr17/synspec_rev1/allStarLite-dr17-synspec_rev1.fits"

SNR_MIN = 70.0
TEFF_MIN, TEFF_MAX = 3500.0, 7000.0
LOGG_MIN, LOGG_MAX = 0.0, 5.0
BATCH_SIZE = 500
DOWNLOAD_XP = False  # Change to True only after the metadata checks pass.

print("Project root:", PROJECT_ROOT)
print("allStarLite path:", ALLSTAR_PATH)

The current archive release 3.15 contains two breaking changes. Please see https://www.cosmos.esa.int/web/gaia/news#GACSrelease315 for details. In case of questions or feedback, please consult the Gaia helpdesk at https://www.cosmos.esa.int/web/gaia/gaia-helpdesk.
Project root: /Users/hmorg/CS4425/group
allStarLite path: /Users/hmorg/CS4425/group/data/allStarLite-dr17-synspec_rev1.fits


## Section 2: obtain and inspect allStarLite

The notebook uses the DR17 `synspec_rev1` allStarLite catalog. The download is explicit and resumable at the file level: an existing nonempty file is not replaced.

In [2]:
# Download only when the file is absent.
if not ALLSTAR_PATH.exists() or ALLSTAR_PATH.stat().st_size == 0:
    from urllib.request import urlretrieve
    print("Downloading", ALLSTAR_URL)
    urlretrieve(ALLSTAR_URL, ALLSTAR_PATH)
else:
    print("Using existing file:", ALLSTAR_PATH)

with fits.open(ALLSTAR_PATH, memmap=True) as hdul:
    print(hdul.info())
    allstar = Table(hdul[1].data)

print("Loaded rows:", len(allstar))
print("Columns:", len(allstar.colnames))

Using existing file: /Users/hmorg/CS4425/group/data/allStarLite-dr17-synspec_rev1.fits
Filename: /Users/hmorg/CS4425/group/data/allStarLite-dr17-synspec_rev1.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       5   ()      
  1                1 BinTableHDU    399   733901R x 191C   [30A, 6A, 20A, 30A, D, D, D, D, E, E, E, E, E, E, E, 32A, E, E, J, J, J, J, J, J, 132A, 32A, 32A, J, E, E, K, 132A, K, 132A, E, E, E, E, E, E, E, E, E, E, E, J, J, E, E, E, E, E, E, K, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, 8A, E, 9K, K, 256A, E, E, E, 27E, 27E, 27E, 27E, 27E, 27E, J, K, 10A, 27E, 27E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, J, E, E, E, J, E, E, E, J, E, E, E, J, E, E, E, J, E, E, E, J, E, E, E, J, E, E, E, J, E, E, E, J, E, E, E, J, E, E, E, J, E, E, E, J, E, E, E, J, E, E, E, J, E, E, E, J, E, E, E, J, E, E, E, J, E, E, E, J, E, E, E, J, E, E, E, J, E, E, E, J, E, E, E, J]   
  2                1 ImageHDU         6   (

None
Loaded rows: 733901
Columns: 191


In [3]:
# Resolve required columns defensively because FITS readers preserve DR17 names exactly.
REQUIRED = [
    "APOGEE_ID", "RA", "DEC", "SNR", "ASPCAPFLAG", "STARFLAG",
    "TEFF", "LOGG", "FE_H", "N_FE", "O_FE", "AL_FE"
]
missing = [c for c in REQUIRED if c not in allstar.colnames]
if missing:
    raise KeyError(f"Missing required DR17 columns: {missing}")

GAIA_ID_CANDIDATES = ["GAIAEDR3_SOURCE_ID", "GAIA_SOURCE_ID", "GAIA_DR3_SOURCE_ID"]
gaia_id_col = next((c for c in GAIA_ID_CANDIDATES if c in allstar.colnames), None)
print("Gaia ID column found:", gaia_id_col)
print("Target columns:", ["TEFF", "LOGG", "FE_H", "N_FE", "O_FE", "AL_FE"])

Gaia ID column found: GAIAEDR3_SOURCE_ID
Target columns: ['TEFF', 'LOGG', 'FE_H', 'N_FE', 'O_FE', 'AL_FE']


## Section 3: define labels and flags

The fifth target is **[N/O]**, computed as `[N/Fe] - [O/Fe]`. APOGEE DR17 supplies calibrated `N_FE` and `O_FE`; there is no need to treat `N_M` as N/O. The notebook retains `AL_FE` directly.

The baseline quality policy rejects `STAR_BAD`, requires
`SNR >= 70`, requires all five target labels to be finite,
and restricts temperature and surface gravity to the selected
ranges. After Gaia DR3 verification and the XP-availability
check, the final quality filter removes sources carrying
`TEFF_WARN`, `LOGG_WARN`, `M_H_WARN`, or `N_M_WARN`.
`VMICRO_WARN` is recorded but is not used as an exclusion
because microturbulence is not a prediction target.

In [4]:
# Convert selected FITS columns to a pandas DataFrame.
cols = REQUIRED.copy()
if gaia_id_col:
    cols.append(gaia_id_col)
for optional in ["PARAMFLAG", "ELEMFLAG"]:
    if optional in allstar.colnames:
        cols.append(optional)

df = allstar[cols].to_pandas()
df["N_O"] = pd.to_numeric(df["N_FE"], errors="coerce") - pd.to_numeric(df["O_FE"], errors="coerce")
TARGETS = ["TEFF", "LOGG", "FE_H", "N_O", "AL_FE"]

ASPCAP_STAR_BAD_BIT = 23
BAD_MEASUREMENT_MASK = sum(1 << bit for bit in range(8))

def bit_is_set(values, bit):
    values = pd.to_numeric(values, errors="coerce").fillna(0).astype("int64")
    return (values & (1 << bit)) != 0

print("Rows before cuts:", len(df))
print(df[TARGETS].describe().T)

Rows before cuts: 733901
          count         mean          std          min          25%  \
TEFF   689024.0  4994.287109  1338.033569  3088.049805  4377.779297   
LOGG   689024.0     3.082897     1.234761    -0.482825     2.278944   
FE_H   647042.0    -0.231800     0.377003    -2.468600    -0.402030   
N_O    587565.0     0.025660     0.280316    -1.500963    -0.126470   
AL_FE  572099.0     0.084597     0.163549    -2.566144     0.017449   

               50%          75%           max  
TEFF   4792.267578  5214.606445  19869.988281  
LOGG      2.984478     4.314916      5.391378  
FE_H     -0.171510     0.014171      0.957890  
N_O       0.042976     0.171543      2.678937  
AL_FE     0.087211     0.171701      2.931121  


## Section 4: Apply APOGEE quality and label cuts

In [5]:
# Section 4: sequential cuts with counts produced from this run.
cut_log = []
current = df.copy()

def apply_cut(name, mask):
    global current
    before = len(current)
    mask = pd.Series(mask, index=current.index).fillna(False)
    current = current.loc[mask].copy()
    cut_log.append({"cut": name, "before": before, "after": len(current), "removed": before-len(current)})
    print(f"{name}: {before:,} -> {len(current):,} (removed {before-len(current):,})")

apply_cut("ASPCAP result is not STAR_BAD", ~bit_is_set(current["ASPCAPFLAG"], ASPCAP_STAR_BAD_BIT))
apply_cut(f"SNR >= {SNR_MIN:g}", pd.to_numeric(current["SNR"], errors="coerce") >= SNR_MIN)
apply_cut("all five targets finite", np.isfinite(current[TARGETS].apply(pd.to_numeric, errors="coerce")).all(axis=1))
apply_cut(f"{TEFF_MIN:g} <= TEFF <= {TEFF_MAX:g}", current["TEFF"].between(TEFF_MIN, TEFF_MAX))
apply_cut(f"{LOGG_MIN:g} <= LOGG <= {LOGG_MAX:g}", current["LOGG"].between(LOGG_MIN, LOGG_MAX))

cut_log_df = pd.DataFrame(cut_log)
cut_log_df.to_csv(DATA_DIR / "apogee_training_cutflow.csv", index=False)
display(cut_log_df)

ASPCAP result is not STAR_BAD: 733,901 -> 691,591 (removed 42,310)
SNR >= 70: 691,591 -> 597,478 (removed 94,113)
all five targets finite: 597,478 -> 495,611 (removed 101,867)
3500 <= TEFF <= 7000: 495,611 -> 495,611 (removed 0)
0 <= LOGG <= 5: 495,611 -> 495,464 (removed 147)


,cut,before,after,removed
0,ASPCAP result is not STAR_BAD,733901,691591,42310
1,SNR >= 70,691591,597478,94113
2,all five targets finite,597478,495611,101867
3,3500 <= TEFF <= 7000,495611,495611,0
4,0 <= LOGG <= 5,495611,495464,147


### Individual abundance/parameter flags

DR17 stores parameter and element flags as arrays in some allStar products. Because their array ordering must match the file's documented parameter/element ordering, this notebook reports their presence rather than guessing indices. The conservative `STAR_BAD` filter above is always applied. Before the final science sample is frozen, inspect the file header/data model and add per-label BAD-bit filtering if the array ordering is present and verified.

In [6]:
print("PARAMFLAG present:", "PARAMFLAG" in allstar.colnames)
print("ELEMFLAG present:", "ELEMFLAG" in allstar.colnames)
print("ASPCAP STAR_WARN count in retained sample:", int(bit_is_set(current["ASPCAPFLAG"], 7).sum()))
print("ASPCAP STAR_BAD count in retained sample:", int(bit_is_set(current["ASPCAPFLAG"], 23).sum()))

def summarize_aspcap_warnings(frame, sample_name):
    counts = pd.Series({
        "TEFF_WARN": bit_is_set(frame["ASPCAPFLAG"], 0).sum(),
        "LOGG_WARN": bit_is_set(frame["ASPCAPFLAG"], 1).sum(),
        "VMICRO_WARN": bit_is_set(frame["ASPCAPFLAG"], 2).sum(),
        "M_H_WARN": bit_is_set(frame["ASPCAPFLAG"], 3).sum(),
        "N_M_WARN": bit_is_set(frame["ASPCAPFLAG"], 6).sum(),
        "STAR_WARN": bit_is_set(frame["ASPCAPFLAG"], 7).sum(),
        "STAR_BAD": bit_is_set(frame["ASPCAPFLAG"], 23).sum(),
    })

    summary = pd.DataFrame({
        "count": counts,
        "percent": 100 * counts / len(frame)
    })

    print(f"{sample_name}: {len(frame):,} rows")
    return summary

PARAMFLAG present: True
ELEMFLAG present: False
ASPCAP STAR_WARN count in retained sample: 32840
ASPCAP STAR_BAD count in retained sample: 0


## Section 5: Gaia DR3 source IDs

Preferred strategy: use the catalog's Gaia EDR3 source ID directly, because Gaia EDR3 and Gaia DR3 use the same source identifiers. Then verify every ID against `gaiadr3.gaia_source` and require `has_xp_continuous = TRUE`. If the catalog lacks a Gaia ID column, stop rather than silently performing an ambiguous positional match; a proper-motion-aware Gaia Archive crossmatch should then be added as a separate audited step.

In [7]:
if gaia_id_col is None:
    raise RuntimeError(
        "No Gaia source-ID column was found. Add a proper-motion-aware positional crossmatch; "
        "do not continue with nearest-neighbour matching silently."
    )

current["source_id"] = pd.to_numeric(current[gaia_id_col], errors="coerce").astype("Int64")
apply_cut("valid positive Gaia source_id", current["source_id"].notna() & (current["source_id"] > 0))
apply_cut("one row per Gaia source_id", ~current["source_id"].duplicated(keep="first"))

upload = Table.from_pandas(current[["source_id"]].astype({"source_id":"int64"}))
upload_path = DATA_DIR / "apogee_gaia_ids.vot"
upload.write(upload_path, format="votable", overwrite=True)
print("Upload table rows:", len(upload))

valid positive Gaia source_id: 495,464 -> 494,613 (removed 851)
one row per Gaia source_id: 494,613 -> 448,728 (removed 45,885)
Upload table rows: 448728


# Section 6: verify IDs and XP availability in Gaia DR3

In [8]:
# Section 6: verify Gaia IDs and XP availability in chunks.

from astropy.table import Table
import pandas as pd
import time

GAIA_QUERY_BATCH_SIZE = 40_000
GAIA_QUERY_PAUSE = 2
GAIA_QUERY_RETRIES = 3

gaia_query_dir = DATA_DIR / "apogee_gaia_query_batches"
gaia_query_dir.mkdir(exist_ok=True)

Gaia.ROW_LIMIT = -1


def query_gaia_batch(batch_df, batch_number):
    """
    Upload one source-ID batch and join it to gaiadr3.gaia_source.
    Existing completed CSV files are reused when the notebook resumes.
    """
    batch_output = (
        gaia_query_dir /
        f"gaia_xp_check_{batch_number:04d}.csv"
    )

    # Resume from an existing successful batch.
    if batch_output.exists() and batch_output.stat().st_size > 0:
        saved = pd.read_csv(
            batch_output,
            dtype={"source_id": "int64"}
        )

        if len(saved) > len(batch_df):
            raise RuntimeError(
                f"{batch_output.name} contains more rows than its input batch."
            )

        print(
            f"Batch {batch_number}: reused "
            f"{len(saved):,} existing rows"
        )
        return saved

    batch_upload = Table.from_pandas(
        batch_df[["source_id"]].astype({"source_id": "int64"})
    )

    batch_upload_path = (
        DATA_DIR /
        f"apogee_gaia_ids_batch_{batch_number:04d}.vot"
    )

    batch_upload.write(
        batch_upload_path,
        format="votable",
        overwrite=True
    )

    adql = """
    SELECT
        u.source_id,
        g.has_xp_continuous
    FROM tap_upload.apogee_ids AS u
    JOIN gaiadr3.gaia_source AS g
        ON g.source_id = u.source_id
    """

    last_error = None

    for attempt in range(1, GAIA_QUERY_RETRIES + 1):
        try:
            print(
                f"Batch {batch_number}, attempt {attempt}: "
                f"submitting {len(batch_df):,} IDs"
            )

            job = Gaia.launch_job_async(
                adql,
                upload_resource=str(batch_upload_path),
                upload_table_name="apogee_ids",
                dump_to_file=False,
            )

            result = job.get_results().to_pandas()

            result["source_id"] = pd.to_numeric(
                result["source_id"],
                errors="raise"
            ).astype("int64")

            # A one-to-one source-ID join cannot return more rows
            # than were submitted.
            if len(result) > len(batch_df):
                raise RuntimeError(
                    f"Batch {batch_number} returned {len(result):,} rows "
                    f"for only {len(batch_df):,} submitted IDs."
                )

            if len(result) == 50_000:
                raise RuntimeError(
                    f"Batch {batch_number} reached the 50,000-row cap."
                )

            if result["source_id"].duplicated().any():
                raise RuntimeError(
                    f"Batch {batch_number} returned duplicate source IDs."
                )

            result.to_csv(batch_output, index=False)

            print(
                f"Batch {batch_number}: "
                f"{len(result):,} IDs verified, "
                f"{int(result['has_xp_continuous'].fillna(False).sum()):,} "
                f"with XP"
            )

            return result

        except Exception as exc:
            last_error = exc
            print(
                f"Batch {batch_number}, attempt {attempt} failed: {exc}"
            )

            if attempt < GAIA_QUERY_RETRIES:
                time.sleep(30 * attempt)

    raise RuntimeError(
        f"Batch {batch_number} failed after "
        f"{GAIA_QUERY_RETRIES} attempts."
    ) from last_error

# Ensure the submitted source IDs are unique.
query_ids = (
    current[["source_id"]]
    .dropna()
    .copy()
)

query_ids["source_id"] = query_ids["source_id"].astype("int64")
query_ids = query_ids.drop_duplicates("source_id").reset_index(drop=True)

print("Total unique IDs to submit:", len(query_ids))

gaia_batch_results = []

for start in range(0, len(query_ids), GAIA_QUERY_BATCH_SIZE):
    stop = min(start + GAIA_QUERY_BATCH_SIZE, len(query_ids))
    batch_number = start // GAIA_QUERY_BATCH_SIZE

    batch_df = query_ids.iloc[start:stop].copy()

    result = query_gaia_batch(
        batch_df=batch_df,
        batch_number=batch_number
    )

    gaia_batch_results.append(result)

    if stop < len(query_ids):
        time.sleep(GAIA_QUERY_PAUSE)

gaia_check = pd.concat(
    gaia_batch_results,
    ignore_index=True
)

gaia_check["source_id"] = pd.to_numeric(
    gaia_check["source_id"],
    errors="raise"
).astype("int64")

print("Combined Gaia rows:", len(gaia_check))
print(
    "Unique combined Gaia IDs:",
    gaia_check["source_id"].nunique()
)
print(
    "Duplicate combined Gaia IDs:",
    gaia_check["source_id"].duplicated().sum()
)
print(
    "Combined IDs with XP continuous:",
    int(
        gaia_check["has_xp_continuous"]
        .fillna(False)
        .sum()
    )
)

if gaia_check["source_id"].duplicated().any():
    duplicate_ids = gaia_check.loc[
        gaia_check["source_id"].duplicated(keep=False),
        "source_id"
    ]

    raise RuntimeError(
        f"The combined Gaia result contains "
        f"{duplicate_ids.nunique():,} duplicated source IDs."
    )

unexpected_ids = set(gaia_check["source_id"]) - set(query_ids["source_id"])

if unexpected_ids:
    raise RuntimeError(
        f"Gaia returned {len(unexpected_ids):,} IDs "
        f"that were not submitted."
    )

submitted_ids = set(query_ids["source_id"])
verified_ids = set(gaia_check["source_id"])

unverified_ids = sorted(submitted_ids - verified_ids)

print("Submitted IDs:", len(submitted_ids))
print("Verified Gaia DR3 IDs:", len(verified_ids))
print("IDs not returned:", len(unverified_ids))

pd.DataFrame({
    "source_id": unverified_ids
}).to_csv(
    DATA_DIR / "apogee_gaia_unverified_ids.csv",
    index=False
)

training_meta = current.merge(
    gaia_check,
    on="source_id",
    how="inner",
    validate="one_to_one"
)

training_meta = training_meta.loc[
    training_meta["has_xp_continuous"].fillna(False)
].copy()

print("Rows after APOGEE and source-ID cuts:", len(current))
print("Unique IDs submitted:", len(query_ids))
print("IDs verified in Gaia DR3:", len(gaia_check))
print("IDs not returned by Gaia DR3:", len(unverified_ids))
print("Rows with XP continuous:", len(training_meta))
print(
    "Unique training source IDs:",
    training_meta["source_id"].nunique()
)
print(
    "Duplicate training source IDs:",
    training_meta["source_id"].duplicated().sum()
)

keep_columns = [
    "source_id",
    "APOGEE_ID",
    "RA",
    "DEC",
    "SNR",
    "TEFF",
    "LOGG",
    "FE_H",
    "N_FE",
    "O_FE",
    "N_O",
    "AL_FE",
    "ASPCAPFLAG",
    "STARFLAG",
]

training_meta[keep_columns].to_csv(
    DATA_DIR /
    "apogee_dr17_gaia_dr3_xp_training_metadata.csv",
    index=False
)

print("Saved complete training metadata:", len(training_meta))

warning_counts = pd.Series({
    "TEFF_WARN": bit_is_set(
        training_meta["ASPCAPFLAG"], 0
    ).sum(),

    "LOGG_WARN": bit_is_set(
        training_meta["ASPCAPFLAG"], 1
    ).sum(),

    "VMICRO_WARN": bit_is_set(
        training_meta["ASPCAPFLAG"], 2
    ).sum(),

    "M_H_WARN": bit_is_set(
        training_meta["ASPCAPFLAG"], 3
    ).sum(),

    "N_M_WARN": bit_is_set(
        training_meta["ASPCAPFLAG"], 6
    ).sum(),

    "STAR_WARN": bit_is_set(
        training_meta["ASPCAPFLAG"], 7
    ).sum(),

    "STAR_BAD": bit_is_set(
        training_meta["ASPCAPFLAG"], 23
    ).sum(),
})

warning_summary = pd.DataFrame({
    "count": warning_counts,
    "percent": (
        100 * warning_counts / len(training_meta)
    )
})

display(warning_summary)



Total unique IDs to submit: 448728
Batch 0: reused 40,000 existing rows
Batch 1: reused 40,000 existing rows
Batch 2: reused 40,000 existing rows
Batch 3: reused 40,000 existing rows
Batch 4: reused 40,000 existing rows
Batch 5: reused 40,000 existing rows
Batch 6: reused 40,000 existing rows
Batch 7: reused 40,000 existing rows
Batch 8: reused 40,000 existing rows
Batch 9: reused 40,000 existing rows
Batch 10: reused 40,000 existing rows
Batch 11: reused 8,728 existing rows
Combined Gaia rows: 448728
Unique combined Gaia IDs: 448728
Duplicate combined Gaia IDs: 0
Combined IDs with XP continuous: 405160
Submitted IDs: 448728
Verified Gaia DR3 IDs: 448728
IDs not returned: 0
Rows after APOGEE and source-ID cuts: 448728
Unique IDs submitted: 448728
IDs verified in Gaia DR3: 448728
IDs not returned by Gaia DR3: 0
Rows with XP continuous: 405160
Unique training source IDs: 405160
Duplicate training source IDs: 0
Saved complete training metadata: 405160


,count,percent
TEFF_WARN,11958,2.951427
LOGG_WARN,3966,0.978873
VMICRO_WARN,92018,22.711521
M_H_WARN,150,0.037022
N_M_WARN,57381,14.162553
STAR_WARN,26521,6.545809
STAR_BAD,0,0.000000


In [9]:
print("IDs verified in Gaia DR3:", len(gaia_check))
print("IDs not returned:", len(unverified_ids))

target_warning_mask = (
    bit_is_set(training_meta["ASPCAPFLAG"], 0) |  # TEFF_WARN
    bit_is_set(training_meta["ASPCAPFLAG"], 1) |  # LOGG_WARN
    bit_is_set(training_meta["ASPCAPFLAG"], 3) |  # M_H_WARN
    bit_is_set(training_meta["ASPCAPFLAG"], 6)    # N_M_WARN
)

print("Current XP sample:", len(training_meta))
print("Rows with a target-related warning:", target_warning_mask.sum())
print("Rows remaining:", (~target_warning_mask).sum())
print(
    "Percent retained:",
    100 * (~target_warning_mask).mean()
)

policies = {
    "Target warnings only": target_warning_mask,

    "Any STAR_WARN": bit_is_set(
        training_meta["ASPCAPFLAG"], 7
    ),

    "Target warnings or STAR_WARN": (
        target_warning_mask |
        bit_is_set(training_meta["ASPCAPFLAG"], 7)
    ),

    "Target warnings or VMICRO_WARN": (
        target_warning_mask |
        bit_is_set(training_meta["ASPCAPFLAG"], 2)
    ),
}

for name, mask in policies.items():
    print(
        f"{name}: remove {mask.sum():,}, "
        f"retain {(~mask).sum():,} "
        f"({100 * (~mask).mean():.2f}%)"
    )

IDs verified in Gaia DR3: 448728
IDs not returned: 0
Current XP sample: 405160
Rows with a target-related warning: 70674
Rows remaining: 334486
Percent retained: 82.55652088063975
Target warnings only: remove 70,674, retain 334,486 (82.56%)
Any STAR_WARN: remove 26,521, retain 378,639 (93.45%)
Target warnings or STAR_WARN: remove 78,921, retain 326,239 (80.52%)
Target warnings or VMICRO_WARN: remove 126,795, retain 278,365 (68.70%)


In [10]:
training_meta_quality = (
    training_meta.loc[~target_warning_mask]
    .copy()
    .reset_index(drop=True)
)

print("Before target-warning filter:", len(training_meta))
print("After target-warning filter:", len(training_meta_quality))
print(
    "Duplicate source IDs:",
    training_meta_quality["source_id"].duplicated().sum()
)

training_meta_quality.to_csv(
    DATA_DIR /
    "apogee_dr17_gaia_dr3_xp_training_metadata_quality.csv",
    index=False
)

final_cutflow = pd.DataFrame([
    {
        "cut": "Valid unique Gaia source ID",
        "before": 495464,
        "after": 448728,
        "removed": 46736,
    },
    {
        "cut": "Verified in Gaia DR3",
        "before": 448728,
        "after": len(gaia_check),
        "removed": 448728 - len(gaia_check),
    },
    {
        "cut": "Gaia XP continuous available",
        "before": len(gaia_check),
        "after": len(training_meta),
        "removed": len(gaia_check) - len(training_meta),
    },
    {
        "cut": "No target-related ASPCAP warnings",
        "before": len(training_meta),
        "after": len(training_meta_quality),
        "removed": len(training_meta) - len(training_meta_quality),
    },
])

complete_cutflow = pd.concat(
    [
        pd.read_csv(DATA_DIR / "apogee_training_cutflow.csv"),
        final_cutflow,
    ],
    ignore_index=True,
)

complete_cutflow.to_csv(
    DATA_DIR / "apogee_training_cutflow_complete.csv",
    index=False
)

display(complete_cutflow)


Before target-warning filter: 405160
After target-warning filter: 334486
Duplicate source IDs: 0


,cut,before,after,removed
0,ASPCAP result is not STAR_BAD,733901,691591,42310
1,SNR >= 70,691591,597478,94113
2,all five targets finite,597478,495611,101867
3,3500 <= TEFF <= 7000,495611,495611,0
4,0 <= LOGG <= 5,495611,495464,147
5,Valid unique Gaia source ID,495464,448728,46736
6,Verified in Gaia DR3,448728,448728,0
7,Gaia XP continuous available,448728,405160,43568
8,No target-related ASPCAP warnings,405160,334486,70674


## Section 7: optional resumable XP download

Each completed batch is stored separately. Rerunning skips existing nonempty FITS batches. HTTP failures are recorded and do not erase completed work.

In [11]:
def download_xp_batches(source_ids, batch_size=BATCH_SIZE, max_retries=3):
    source_ids = [int(x) for x in source_ids]
    failures = []
    for start in range(0, len(source_ids), batch_size):
        batch_num = start // batch_size
        out = BATCH_DIR / f"apogee_xp_{batch_num:05d}.fits"
        if out.exists() and out.stat().st_size > 0:
            print("skip", out.name)
            continue
        ids = source_ids[start:start+batch_size]
        for attempt in range(1, max_retries + 1):
            try:
                result = Gaia.load_data(
                    ids=ids,
                    data_release="Gaia DR3",
                    retrieval_type="XP_CONTINUOUS",
                    data_structure="DATAMODEL_GAIA",
                    format="fits",
                )
                files = result.get("XP_CONTINUOUS_COMBINED.xml", [])
                if not files:
                    raise RuntimeError("Gaia.load_data returned no combined XP file")
                # Astroquery returns temporary file paths; copy the first combined product.
                import shutil
                shutil.copy2(str(files[0]), out)
                print("saved", out.name, "requested", len(ids))
                break
            except Exception as exc:
                print(f"batch {batch_num}, attempt {attempt}: {exc}")
                if attempt == max_retries:
                    failures.append({"batch": batch_num, "start": start, "error": repr(exc)})
                else:
                    time.sleep(30 * attempt)
    pd.DataFrame(failures).to_csv(DATA_DIR / "apogee_xp_download_failures.csv", index=False)
    return failures

if DOWNLOAD_XP:
    failures = download_xp_batches(
        training_meta_quality["source_id"]
    )
    print("Failed batches:", len(failures))
else:
    print("XP download disabled. Set DOWNLOAD_XP = True after reviewing the cutflow and metadata.")

XP download disabled. Set DOWNLOAD_XP = True after reviewing the cutflow and metadata.


## Section 8: Save final audit outputs

Run this after the preceding sections. It writes only observed counts and current configuration values.

In [12]:
quality_path = (
    DATA_DIR /
    "apogee_dr17_gaia_dr3_xp_training_metadata_quality.csv"
)

training_meta_quality.to_csv(
    quality_path,
    index=False
)

print("Saved:", quality_path)
print("Rows:", len(training_meta_quality))

training_meta_quality[["source_id"]].to_csv(
    DATA_DIR / "apogee_training_xp_source_ids.csv",
    index=False
)

complete_cutflow.to_csv(
    DATA_DIR / "apogee_training_cutflow.csv",
    index=False
)

summary = {
    "allstar_rows_loaded": 733901,
    "rows_after_star_bad_cut": 691591,
    "rows_after_snr_cut": 597478,
    "rows_after_finite_target_cut": 495611,
    "rows_after_teff_cut": 495611,
    "rows_after_logg_cut": 495464,
    "valid_unique_gaia_ids": 448728,
    "gaia_dr3_ids_verified": 448728,
    "gaia_dr3_ids_not_returned": 0,
    "rows_with_xp_continuous": 405160,
    "rows_after_target_warning_filter": 334486,
    "duplicate_final_source_ids": 0,
    "snr_min": 70,
    "teff_range": [3500, 7000],
    "logg_range": [0, 5],
    "targets": ["TEFF", "LOGG", "FE_H", "N_O", "AL_FE"],
    "n_o_definition": "N_FE - O_FE",
    "excluded_warning_bits": [
        "TEFF_WARN",
        "LOGG_WARN",
        "M_H_WARN",
        "N_M_WARN"
    ]
}

import json

with open(
    DATA_DIR / "apogee_training_summary.json",
    "w"
) as file:
    json.dump(summary, file, indent=2)

Saved: /Users/hmorg/CS4425/group/data/apogee_dr17_gaia_dr3_xp_training_metadata_quality.csv
Rows: 334486


In [13]:
test_ids = (
    training_meta_quality["source_id"]
    .head(10)
    .astype("int64")
    .tolist()
)

try:
    test_result = Gaia.load_data(
        ids=test_ids,
        data_release="Gaia DR3",
        retrieval_type="XP_CONTINUOUS",
        data_structure="DATAMODEL_GAIA",
        format="fits",
    )

    print("XP test succeeded.")
    print("Returned products:", list(test_result.keys()))

except Exception as error:
    print(
        "XP test unavailable. Gaia DataLink returned an error:",
        type(error).__name__
    )
    print(
        "The metadata and XP-availability results remain valid. "
        "Keep DOWNLOAD_XP = False and retry later."
    )

503 Error 503:
<!DOCTYPE HTML PUBLIC "-//IETF//DTD HTML 2.0//EN">
<html><head>
<title>503 Service Unavailable</title>
</head><body>
<h1>Service Unavailable</h1>
<p>The server is temporarily unable to service your
request due to maintenance downtime or capacity
problems. Please try again later.</p>
</body></html>

XP test unavailable. Gaia DataLink returned an error: HTTPError
The metadata and XP-availability results remain valid. Keep DOWNLOAD_XP = False and retry later.
